In [1]:
import pandas as pd
import numpy as np
import os
import json

# --- Configuration ---
NUM_ITEMS = 600
NUM_ORDERS = 5000 
DATA_DIR = "../data/" 
if not os.path.exists(DATA_DIR): os.makedirs(DATA_DIR)

# 1. Enhanced Menu Items (Added 'Side' for Salan)
categories = {
    "Main": ["Biryani", "Burger", "Pasta", "Pizza", "Thali", "Wrap"],
    "Starter": ["Fries", "Wings", "Garlic Bread", "Roll"],
    "Dessert": ["Ice Cream", "Gulab Jamun", "Brownie"],
    "Beverage": ["Coke", "Cold Coffee", "Shake", "Water"],
    "Side": ["Salan", "Raita", "Extra Dip"] # Specifically added for sequential patterns
}

menu_data = []
for i in range(1, NUM_ITEMS + 1):
    cat = np.random.choice(list(categories.keys()))
    sub_cat = np.random.choice(categories[cat])
    menu_data.append({
        "item_id": f"item_{i}",
        "item_name": f"{np.random.choice(['Spicy', 'Classic', 'Royal'])} {sub_cat} {i}",
        "category": cat,
        "price": np.random.randint(100, 800)
    })

menu_items = pd.DataFrame(menu_data)
menu_items.to_csv(os.path.join(DATA_DIR, "menu_items.csv"), index=False)

# 2. Realistic Order Generation with Pattern Injection
orders_data = []
meal_times = ["Breakfast", "Lunch", "Dinner", "Late Night"]

# Pre-identify Biryani and Salan IDs for injection
biryani_ids = menu_items[menu_items['item_name'].str.contains("Biryani")]['item_id'].tolist()
salan_ids = menu_items[menu_items['item_name'].str.contains("Salan")]['item_id'].tolist()

for order_id in range(NUM_ORDERS):
    time = np.random.choice(meal_times)
    cart_size = np.random.randint(2, 6) if time in ["Lunch", "Dinner"] else np.random.randint(1, 4)
    
    # --- PATTERN INJECTION (The AI Edge) ---
    # We force 40% of Biryani orders to contain Salan so the model learns the sequence
    if time in ["Lunch", "Dinner"] and np.random.random() < 0.4:
        b_id = np.random.choice(biryani_ids)
        s_id = np.random.choice(salan_ids)
        cart = [b_id, s_id]
        # Fill rest of cart randomly
        if cart_size > 2:
            others = np.random.choice(menu_items['item_id'], size=cart_size-2, replace=False)
            cart.extend(list(others))
    else:
        # Standard Random Generation
        probs = menu_items['category'].map({"Main": 0.4, "Starter": 0.2, "Beverage": 0.2, "Dessert": 0.1, "Side": 0.1})
        probs /= probs.sum()
        cart = list(np.random.choice(menu_items['item_id'], size=cart_size, replace=False, p=probs))
    
    orders_data.append({
        "order_id": order_id,
        "meal_time": time,
        "user_segment": np.random.choice(["Budget", "Premium", "Regular"]),
        "cart_items": cart 
    })

orders_df = pd.DataFrame(orders_data)
orders_df['cart_items'] = orders_df['cart_items'].apply(json.dumps)
orders_df.to_csv(os.path.join(DATA_DIR, "orders.csv"), index=False)
print("01_Data: Successfully injected Biryani -> Salan patterns for AI training.")

01_Data: Successfully injected Biryani -> Salan patterns for AI training.
